# 🔐 ARP Spoofing Attack Detection
### Mini-Project — Data Security

**Objective:** Detect ARP Spoofing attacks (Man-in-the-Middle) using ML and DL models.

**Dataset:** [CIC MITM ARP Spoofing Dataset — Kaggle](https://www.kaggle.com/datasets/mizanunswcyber/arp-spoofing-based-mitm-attack-dataset)

---
| Step | Description |
|------|-------------|
| 1 | Loading & exploring the dataset |
| 2 | Preprocessing (cleaning, normalization, encoding, split) |
| 3 | Training: Random Forest, KNN, CNN |
| 4 | Evaluation: Accuracy, Precision, Recall, F1 + graphs |
| 5 | Comparison & conclusion |

## Step 0 — Installation & Imports

In [ ]:
# Installation of necessary libraries
!pip install kaggle scikit-learn tensorflow seaborn matplotlib pandas numpy imbalanced-learn -q
print("SUUUIIIIIIIIII")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2

from imblearn.over_sampling import SMOTE

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
print('✅ All imports successful')

## Step 1 — Loading the Dataset

In [ ]:
from google.colab import files
print('Please upload your kaggle.json file ...')
uploaded = files.upload()

In [ ]:
df = pd.read_csv("CIC_MITM_ArpSpoofing_All_Labelled.csv", low_memory=False)
print(f'\nDimensions: {df.shape}')
print(f'Columns: {df.shape[1]}')
print(f'Rows: {df.shape[0]:,}')
# Dataset overview
print('=== Overview (first 5 rows) ===')
display(df.head())


print('\n=== General information ===')
df.info()

print('\n=== Descriptive statistics ===')
display(df.describe())

In [ ]:
# Class distribution (Label column)
# Identifier la colonne label
label_candidates = [c for c in df.columns if c.lower() in ['label', 'class', 'attack', 'type', 'category']]
LABEL_COL = label_candidates[0] if label_candidates else df.columns[-1]
print(f'Label column identified: "{LABEL_COL}"')

print(f'\n=== Class distribution ===')
class_counts = df[LABEL_COL].value_counts()
print(class_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
colors = ['#2563eb', '#dc2626', '#16a34a', '#d97706', '#7c3aed']
class_counts.plot(kind='bar', ax=axes[0], color=colors[:len(class_counts)], edgecolor='white')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of samples')
axes[0].tick_params(axis='x', rotation=30)
for bar, val in zip(axes[0].patches, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index,
            autopct='%1.1f%%', colors=colors[:len(class_counts)],
            startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph saved: class_distribution.png')

## Step 2 — Preprocessing

In [ ]:
# ─── 2.1 Data cleaning ────────────────────────────────────────────────────────────
print('=== 2.1 Data cleaning ===')
print(f'Missing values before cleaning:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

# Supprimer les doublons
n_before = len(df)
df.drop_duplicates(inplace=True)
print(f'\nDuplicates removed: {n_before - len(df):,}')

# Supprimer les colonnes constantes ou quasi-constantes
# (variance quasi-nulle => peu informatives)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if LABEL_COL in numeric_cols:
    numeric_cols.remove(LABEL_COL)

# Remplacer les inf par NaN
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Remplir les NaN par la médiane (robuste aux outliers)
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

print(f'\nMissing values after cleaning: {df.isnull().sum().sum()}')
print(f'Final dimensions: {df.shape}')

In [ ]:
# ─── 2.2 Label encoding ────────────────────────────────────────
print('=== 2.2 Label encoding ===')

le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df[LABEL_COL].astype(str))

print('Class mapping:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} → {cls}')

N_CLASSES = len(le.classes_)
print(f'\nNumber of classes: {N_CLASSES}')
IS_BINARY = N_CLASSES == 2
print(f'Problem: {"Binary" if IS_BINARY else "Multi-class"}')

In [ ]:
# ─── 2.3 Feature selection ──────────────────────────────────────────────
print('=== 2.3 Feature selection ===')

# Garder uniquement les colonnes numériques (hors label)
drop_cols = [LABEL_COL, 'label_encoded']

# Supprimer aussi les colonnes non numériques (IPs, MACs en format string)
non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
non_numeric = [c for c in non_numeric if c not in drop_cols]
if non_numeric:
    print(f'Non-numeric columns ignored: {non_numeric}')

drop_cols += non_numeric
feature_cols = [c for c in df.columns if c not in drop_cols]

# Supprimer colonnes à variance nulle
zero_var = [c for c in feature_cols if df[c].std() == 0]
if zero_var:
    print(f'Variance=0 columns removed: {zero_var}')
    feature_cols = [c for c in feature_cols if c not in zero_var]

X = df[feature_cols].values
y = df['label_encoded'].values

print(f'\nNumber of features retained: {len(feature_cols)}')
print(f'Shape of X: {X.shape}')
print(f'Shape of y: {y.shape}')

In [ ]:
# ─── 2.4 Correlation analysis (Top features) ──────────────────────────────
print('=== 2.4 Correlation analysis ===')

df_corr = df[feature_cols + ['label_encoded']].copy()
correlations = df_corr.corr()['label_encoded'].drop('label_encoded').abs().sort_values(ascending=False)

top_n = min(20, len(correlations))
top_features = correlations.head(top_n)

plt.figure(figsize=(10, 7))
colors_bar = ['#1e3a8a' if v > 0.5 else '#3b82f6' if v > 0.2 else '#93c5fd' for v in top_features.values]
bars = plt.barh(range(top_n), top_features.values[::-1], color=colors_bar[::-1])
plt.yticks(range(top_n), top_features.index[::-1], fontsize=10)
plt.xlabel('Absolute correlation with the label', fontsize=12)
plt.title(f'Top {top_n} Features most correlated with the label', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold 0.5')
plt.legend()
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 2.5 Normalization ────────────────────────────────────────────────────────
print('=== 2.5 Normalization (StandardScaler) ===')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Mean after scaling (example feature 0): {X_scaled[:, 0].mean():.6f}')
print(f'Standard deviation after scaling (example feature 0): {X_scaled[:, 0].std():.6f}')

In [ ]:
# ─── 2.6 Class balancing (SMOTE) ─────────────────────────
print('=== 2.6 Class balancing (SMOTE) ===')

print('Distribution before SMOTE:')
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  Class {u} ({le.classes_[u]}) : {c:,}')

# Appliquer SMOTE seulement si déséquilibre > 2:1
max_count = max(counts)
min_count = min(counts)
imbalance_ratio = max_count / min_count

if imbalance_ratio > 2:
    print(f'\nImbalance ratio: {imbalance_ratio:.1f} → Applying SMOTE')
    smote = SMOTE(random_state=42)
    X_balanced, y_balanced = smote.fit_resample(X_scaled, y)
    print('Distribution after SMOTE:')
    unique, counts = np.unique(y_balanced, return_counts=True)
    for u, c in zip(unique, counts):
        print(f'  Class {u} ({le.classes_[u]}) : {c:,}')
else:
    print(f'\nImbalance ratio: {imbalance_ratio:.1f} → SMOTE not necessary')
    X_balanced, y_balanced = X_scaled, y

In [ ]:
# ─── 2.7 Train / Test Split ────────────────────────────────────────────────────
print('=== 2.7 Train / Test Split (80/20) ===')

X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced,
    test_size=0.2,
    random_state=42,
    stratify=y_balanced
)

print(f'Training: {X_train.shape[0]:,} samples')
print(f'Test     : {X_test.shape[0]:,} samples')
print(f'Features : {X_train.shape[1]}')

# Pour le MLP : one-hot encoding de la sortie
y_train_cat = to_categorical(y_train, num_classes=N_CLASSES)
y_test_cat  = to_categorical(y_test,  num_classes=N_CLASSES)

print('\n✅ Preprocessing completed — data ready for training')

## Step 3.1 — Random Forest

In [ ]:
print('=== Training: Random Forest ===')

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
    class_weight='balanced'
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)

# Métriques
acc_rf  = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf, average='weighted', zero_division=0)
rec_rf  = recall_score(y_test, y_pred_rf, average='weighted', zero_division=0)
f1_rf   = f1_score(y_test, y_pred_rf, average='weighted', zero_division=0)

print(f'\n📊 Random Forest Results:')
print(f'  Accuracy  : {acc_rf:.4f} ({acc_rf*100:.2f}%)')
print(f'  Precision : {prec_rf:.4f}')
print(f'  Recall    : {rec_rf:.4f}')
print(f'  F1-score  : {f1_rf:.4f}')

print('\n=== Classification report ===')
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

In [ ]:
# Matrice de confusion — Random Forest
def plot_confusion_matrix(y_true, y_pred, labels, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels,
                linewidths=0.5, linecolor='white')
    plt.title(title, fontsize=14, fontweight='bold', pad=15)
    plt.ylabel('True class', fontsize=12)
    plt.xlabel('Predicted class', fontsize=12)
    plt.xticks(rotation=30, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    return cm

cm_rf = plot_confusion_matrix(
    y_test, y_pred_rf, le.classes_,
    'Confusion Matrix — Random Forest',
    'cm_random_forest.png'
)

In [ ]:
# Importance des features — Random Forest
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False)
top_feat = feat_imp.head(20)

plt.figure(figsize=(10, 7))
colors_imp = plt.cm.YlOrRd(np.linspace(0.3, 1.0, len(top_feat)))
bars = plt.barh(range(len(top_feat)), top_feat.values[::-1], color=colors_imp[::-1])
plt.yticks(range(len(top_feat)), top_feat.index[::-1], fontsize=10)
plt.xlabel('Importance (Gini)', fontsize=12)
plt.title('Top 20 Important Features — Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance_rf.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3.2 — LightGBM (Light Gradient Boosting)

In [ ]:
!pip install lightgbm -q
import lightgbm as lgb
print('✅ LightGBM installed')

In [ ]:
print('=== Training: LightGBM ===')
print('─' * 50)

lgb_model = lgb.LGBMClassifier(
    n_estimators      = 300,
    max_depth         = 8,
    learning_rate     = 0.1,
    num_leaves        = 63,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    min_child_samples = 20,
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(30, verbose=False),
               lgb.log_evaluation(50)]
)

y_pred_lgb = lgb_model.predict(X_test)
y_prob_lgb = lgb_model.predict_proba(X_test)

acc_lgb  = accuracy_score(y_test, y_pred_lgb)
prec_lgb = precision_score(y_test, y_pred_lgb, average='weighted', zero_division=0)
rec_lgb  = recall_score(y_test, y_pred_lgb, average='weighted', zero_division=0)
f1_lgb   = f1_score(y_test, y_pred_lgb, average='weighted', zero_division=0)

print(f'\n📊 LightGBM Results:')
print(f'  Accuracy  : {acc_lgb:.4f} ({acc_lgb*100:.2f}%)')
print(f'  Precision : {prec_lgb:.4f}')
print(f'  Recall    : {rec_lgb:.4f}')
print(f'  F1-score  : {f1_lgb:.4f}')

print('\n=== Classification report ===')
print(classification_report(y_test, y_pred_lgb, target_names=le.classes_))

In [ ]:
# Matrice de confusion — LightGBM
cm_lgb = confusion_matrix(y_test, y_pred_lgb)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_lgb, annot=True, fmt='d', cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.5, linecolor='white')
plt.title('Confusion Matrix — LightGBM', fontsize=14, fontweight='bold')
plt.ylabel('True class'); plt.xlabel('Predicted class')
plt.xticks(rotation=30, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('cm_lightgbm.png', dpi=150, bbox_inches='tight')
plt.show()

# Importance des features — LightGBM
imp_lgb = pd.Series(lgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
imp_lgb.head(20)[::-1].plot(kind='barh', color='#16a34a')
plt.title('Top 20 Features — LightGBM', fontsize=13, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance_lgb.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3.3 — MLP (Deep Learning)

In [ ]:
print('=== Building the Improved MLP ===')
print('─' * 50)

# One-hot encoding si pas déjà fait
y_train_cat = to_categorical(y_train, num_classes=N_CLASSES)
y_test_cat  = to_categorical(y_test,  num_classes=N_CLASSES)

n_features = X_train.shape[1]

def build_mlp_v2(n_features, n_classes):
    """
    Improved MLP — Deeper architecture with residual-like connections
    ┌──────────────────────────────────────────────────────────────┐
    │  Input          (n_features,)                                │
    │  Dense 1024     ReLU + BN + Dropout 0.45                    │
    │  Dense 512      ReLU + BN + Dropout 0.40                    │
    │  Dense 256      ReLU + BN + Dropout 0.35                    │
    │  Dense 128      ReLU + BN + Dropout 0.30                    │
    │  Dense 64       ReLU + Dropout 0.20                         │
    │  Dense 32       ReLU                                        │
    │  Output         Softmax                                     │
    └──────────────────────────────────────────────────────────────┘
    """
    model = Sequential([
        # Layer 1 — large
        Dense(1024, activation='relu',
              kernel_regularizer=l2(1e-4),
              input_shape=(n_features,)),
        BatchNormalization(),
        Dropout(0.45),

        # Layer 2
        Dense(512, activation='relu', kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(0.40),

        # Layer 3
        Dense(256, activation='relu', kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(0.35),

        # Layer 4
        Dense(128, activation='relu', kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(0.30),

        # Layer 5
        Dense(64, activation='relu'),
        Dropout(0.20),

        # Layer 6
        Dense(32, activation='relu'),

        # Output
        Dense(n_classes, activation='softmax' if n_classes > 2 else 'sigmoid')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=5e-4,
            beta_1=0.9, beta_2=0.999
        ),
        loss='categorical_crossentropy' if n_classes > 2 else 'binary_crossentropy',
        metrics=['accuracy']
    )
    return model

mlp_v2 = build_mlp_v2(n_features, N_CLASSES)
mlp_v2.summary()
print(f'\nTotal parameters: {mlp_v2.count_params():,}')

In [ ]:
print('=== Training Improved MLP ===')

callbacks_v2 = [
    EarlyStopping(
        monitor='val_loss', patience=15,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.3,
        patience=7, min_lr=1e-7, verbose=1
    )
]

history_v2 = mlp_v2.fit(
    X_train, y_train_cat,
    epochs=80,
    batch_size=512,
    validation_split=0.2,
    callbacks=callbacks_v2,
    verbose=1
)

# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_v2.history['loss'],     label='Train', color='#7c3aed', lw=2)
axes[0].plot(history_v2.history['val_loss'], label='Val',   color='#dc2626', lw=2, ls='--')
axes[0].set_title('Loss — Improved MLP', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history_v2.history['accuracy'],     label='Train', color='#059669', lw=2)
axes[1].plot(history_v2.history['val_accuracy'], label='Val',   color='#d97706', lw=2, ls='--')
axes[1].set_title('Accuracy — Improved MLP', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('mlp_v2_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Evaluation
y_prob_mlp2 = mlp_v2.predict(X_test)
y_pred_mlp2 = np.argmax(y_prob_mlp2, axis=1)

acc_mlp2  = accuracy_score(y_test, y_pred_mlp2)
prec_mlp2 = precision_score(y_test, y_pred_mlp2, average='weighted', zero_division=0)
rec_mlp2  = recall_score(y_test, y_pred_mlp2, average='weighted', zero_division=0)
f1_mlp2   = f1_score(y_test, y_pred_mlp2, average='weighted', zero_division=0)

print(f'\n📊 Improved MLP Results:')
print(f'  Accuracy  : {acc_mlp2:.4f} ({acc_mlp2*100:.2f}%)')
print(f'  Precision : {prec_mlp2:.4f}')
print(f'  Recall    : {rec_mlp2:.4f}')
print(f'  F1-score  : {f1_mlp2:.4f}')
print(f'\n  vs MLP v1 : {(f1_mlp2 - 0.9334)*100:+.2f}%')
print(f'  vs RF     : {(f1_mlp2 - 0.9573)*100:+.2f}%')

print('\n=== Classification report ===')
print(classification_report(y_test, y_pred_mlp2, target_names=le.classes_))

In [ ]:
cm_mlp2 = plot_confusion_matrix(
    y_test, y_pred_mlp2, le.classes_,
    'Confusion Matrix — Improved MLP',
    'cm_mlp_ameliore.png'
)

## 📊 Step 4 — Evaluation & Comparative Graphs

In [ ]:
# ─── 4.1 Performance summary table ────────────────────────────────────────────────
print('=== Performance summary table ===')

results = pd.DataFrame({
    'Model'   : ['Random Forest', 'LightGBM', 'Improved MLP'],
    'Type'     : ['ML', 'ML', 'DL'],
    'Accuracy' : [acc_rf,  acc_lgb,  acc_mlp2],
    'Precision': [prec_rf, prec_lgb, prec_mlp2],
    'Recall'   : [rec_rf,  rec_lgb,  rec_mlp2],
    'F1-Score' : [f1_rf,   f1_lgb,   f1_mlp2]
})

results_display = results.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    results_display[col] = results_display[col].apply(lambda x: f'{x:.4f} ({x*100:.2f}%)')

display(results_display.set_index('Model'))

best_model_idx  = results['F1-Score'].idxmax()
best_model_name = results.loc[best_model_idx, 'Model']
best_f1         = results.loc[best_model_idx, 'F1-Score']
print(f'\n🏆 Best model: {best_model_name} (F1={best_f1:.4f})')

In [ ]:
# ─── 4.4 ROC Curves ──────────────────────────────────────────────────────────
print('=== ROC Curves ===')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
model_data = [
    ('Random Forest', y_prob_rf,   '#2563eb', axes[0]),
    ('LightGBM',      y_prob_lgb,  '#16a34a', axes[1]),
    ('Improved MLP',  y_prob_mlp2, '#7c3aed', axes[2]),
]

for model_name, y_prob, color, ax in model_data:
    if N_CLASSES == 2:
        fpr, tpr, _ = roc_curve(y_test, y_prob[:, 1])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linewidth=2.5,
                label=f'AUC = {roc_auc:.4f}')
    else:
        from sklearn.preprocessing import label_binarize
        y_test_bin = label_binarize(y_test, classes=range(N_CLASSES))
        fpr, tpr, _ = roc_curve(y_test_bin.ravel(), y_prob.ravel())
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linewidth=2.5,
                label=f'AUC micro = {roc_auc:.4f}')

    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, linewidth=1)
    ax.fill_between(fpr, tpr, alpha=0.1, color=color)
    ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(f'ROC Curve — {model_name}', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=11)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5 — Comparison & Conclusion

In [ ]:
# ─── 5.1 Final summary report ────────────────────────────────────────────
print('=' * 70)
print('               FINAL SUMMARY REPORT')
print('         ARP Spoofing Attack Detection (MitM)')
print('=' * 70)

print(f'\n📁 Dataset     : CIC MITM ARP Spoofing Dataset')
print(f'🔖 Classes     : {N_CLASSES} ({" | ".join(le.classes_)})')
print(f'🔢 Features    : {len(feature_cols)}')
print(f'📊 Samples (after SMOTE) : {len(X_balanced):,}')
print(f'   → Train : {len(X_train):,} | Test : {len(X_test):,}')

print('\n' + '─' * 70)
print(f'{"Model":<20} {"Type":<5} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} {"F1-Score":>10}')
print('─' * 70)
medals = ['🥇','🥈','🥉']
results_sorted = results.sort_values('F1-Score', ascending=False).reset_index(drop=True)
for i, row in results_sorted.iterrows():
    print(f'{medals[i]}  {row["Model"]:<18} {row["Type"]:<5} '
          f'{row["Accuracy"]:>10.4f} {row["Precision"]:>10.4f} '
          f'{row["Recall"]:>10.4f} {row["F1-Score"]:>10.4f}')
print('─' * 70)

print(f'\n🏆 Best overall model: {best_model_name}')
print(f'   F1-Score: {best_f1:.4f} ({best_f1*100:.2f}%)')

best_ml = results[results['Type']=='ML'].sort_values('F1-Score',ascending=False).iloc[0]
best_dl = results[results['Type']=='DL'].sort_values('F1-Score',ascending=False).iloc[0]
print(f'\n🌲 Best ML: {best_ml["Model"]} — F1={best_ml["F1-Score"]*100:.2f}%')
print(f'🧠 Best DL: {best_dl["Model"]} — F1={best_dl["F1-Score"]*100:.2f}%')

print('\n' + '=' * 70)
print('COMPARATIVE ANALYSIS')
print('=' * 70)
analysis = """
1. RANDOM FOREST (ML — Ensemble)
   ✅ Robust, fast to train, very interpretable
   ✅ Feature importance available to explain decisions
   ✅ Handles imbalanced data well with class_weight='balanced'
   ❌ Less accurate than LightGBM on tabular data
   ❌ Can overfit on small datasets

2. LIGHTGBM (ML — Gradient Boosting)
   ✅ Champion of tabular data — generally the best accuracy
   ✅ Very fast thanks to histogram-based algorithm
   ✅ Supports early stopping to avoid overfitting
   ✅ Detailed feature importance (gain, split, cover)
   ❌ More hyperparameters to tune than Random Forest
   ❌ Less interpretable than a simple tree

3. IMPROVED MLP (DL — Deep Learning)
   ✅ Captures very complex non-linear relationships
   ✅ Deep architecture (1024→512→256→128→64→32) with L2 regularization
   ✅ BatchNormalization stabilizes training
   ❌ Slower to train than ML models
   ❌ Black box — less interpretable
   ❌ Requires more data to generalize perfectly
"""
print(analysis)

In [ ]:
# ─── 5.3 Saving models ───────────────────────────────────────────────
import joblib, os

os.makedirs('./models', exist_ok=True)

joblib.dump(rf_model,  './models/random_forest.pkl')
joblib.dump(lgb_model, './models/lightgbm.pkl')
joblib.dump(scaler,    './models/scaler.pkl')
joblib.dump(le,        './models/label_encoder.pkl')

mlp_v2.save('./models/mlp_ameliore.h5')

print('✅ Models saved in ./models/')
print('   - random_forest.pkl')
print('   - lightgbm.pkl')
print('   - mlp_ameliore.h5')
print('   - scaler.pkl')
print('   - label_encoder.pkl')

In [ ]:
import zipfile, os
from google.colab import files

os.makedirs('./models', exist_ok=True)

# Zip all models
with zipfile.ZipFile('models.zip', 'w') as zf:
    for f in ['random_forest.pkl', 'lightgbm.pkl',
              'mlp_ameliore.h5', 'scaler.pkl', 'label_encoder.pkl']:
        path = f'./models/{f}'
        if os.path.exists(path):
            zf.write(path, f)
            print(f'✅ Added: {f}')

files.download('models.zip')
print('ZIP downloaded!')

##  Step 6 — Test on Real Traffic (CICFlowMeter CSV)
> Load a CSV file captured with **CICFlowMeter** and detect ARP Spoofing attacks with the 3 trained models.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 6 — Detection on real CSV file (CICFlowMeter)
# ═══════════════════════════════════════════════════════════════════
import os
import pandas as pd
import numpy as np
from google.colab import files as colab_files
import joblib
import tensorflow as tf
import matplotlib.pyplot as plt

# Charger les modèles (adapter les chemins si besoin)
rf_model = joblib.load('./models/random_forest.pkl')
lgb_model = joblib.load('./models/lightgbm.pkl')
mlp_v2 = tf.keras.models.load_model('./models/mlp_ameliore.h5')
scaler = joblib.load('./models/scaler.pkl')
le = joblib.load('./models/label_encoder.pkl')

# Liste des 68 features utilisées lors de l'entraînement
feature_cols = [
    'id', 'src_port', 'dst_port', 'protocol', 'ip_version', 'tunnel_id',
    'bidirectional_first_seen_ms', 'bidirectional_last_seen_ms',
    'bidirectional_duration_ms', 'bidirectional_packets', 'bidirectional_bytes',
    'src2dst_first_seen_ms', 'src2dst_last_seen_ms', 'src2dst_duration_ms',
    'src2dst_packets', 'src2dst_bytes', 'dst2src_first_seen_ms',
    'dst2src_last_seen_ms', 'dst2src_duration_ms', 'dst2src_packets',
    'dst2src_bytes', 'bidirectional_min_ps', 'bidirectional_mean_ps',
    'bidirectional_stddev_ps', 'bidirectional_max_ps', 'src2dst_min_ps',
    'src2dst_mean_ps', 'src2dst_stddev_ps', 'src2dst_max_ps',
    'dst2src_min_ps', 'dst2src_mean_ps', 'dst2src_stddev_ps',
    'dst2src_max_ps', 'bidirectional_min_piat_ms', 'bidirectional_mean_piat_ms',
    'bidirectional_stddev_piat_ms', 'bidirectional_max_piat_ms',
    'src2dst_min_piat_ms', 'src2dst_mean_piat_ms', 'src2dst_stddev_piat_ms',
    'src2dst_max_piat_ms', 'dst2src_min_piat_ms', 'dst2src_mean_piat_ms',
    'dst2src_stddev_piat_ms', 'dst2src_max_piat_ms', 'bidirectional_syn_packets',
    'bidirectional_cwr_packets', 'bidirectional_ece_packets',
    'bidirectional_ack_packets', 'bidirectional_psh_packets',
    'bidirectional_rst_packets', 'bidirectional_fin_packets',
    'src2dst_syn_packets', 'src2dst_cwr_packets', 'src2dst_ece_packets',
    'src2dst_ack_packets', 'src2dst_psh_packets', 'src2dst_rst_packets',
    'src2dst_fin_packets', 'dst2src_syn_packets', 'dst2src_cwr_packets',
    'dst2src_ece_packets', 'dst2src_ack_packets', 'dst2src_psh_packets',
    'dst2src_rst_packets', 'dst2src_fin_packets', 'application_is_guessed',
    'application_confidence'
]

# ─── 6.1 Loading the CSV file ──────────────────────────────
print('=' * 60)
print('  DETECTION ON REAL TRAFFIC — CICFlowMeter CSV')
print('=' * 60)

print('\n📂 Upload your CICFlowMeter CSV file...')
uploaded = colab_files.upload()
csv_live = list(uploaded.keys())[0]

df_live = pd.read_csv(csv_live, low_memory=False)
print(f'\n✅ File loaded: {csv_live}')
print(f'   Captured flows: {len(df_live):,}')
print(f'   Columns      : {df_live.shape[1]}')

# ─── 6.2 Column adaptation (MAPPING WITHOUT DUPLICATE) ─────────
print('\n─── Column adaptation ───────────────────────────────')

# Nettoyer les espaces dans les noms de colonnes
df_live.columns = df_live.columns.str.strip()

# Mapping without conflict (one source per target)
col_mapping = {
    'Src Port': 'src_port',
    'Dst Port': 'dst_port',
    'Protocol': 'protocol',
    'Flow Duration': 'bidirectional_duration_ms',
    'Tot Fwd Pkts': 'bidirectional_packets',   # on garde seulement Fwd
    # 'Tot Bwd Pkts' → ignoré
    'TotLen Fwd Pkts': 'bidirectional_bytes',  # on garde seulement Fwd
    # 'TotLen Bwd Pkts' → ignoré
    'Fwd Pkt Len Max': 'bidirectional_max_ps',
    'Fwd Pkt Len Min': 'bidirectional_min_ps',
    'Fwd Pkt Len Mean': 'bidirectional_mean_ps',
    'Fwd Pkt Len Std': 'bidirectional_stddev_ps',
    'Bwd Pkt Len Max': 'bidirectional_max_ps',
    'Bwd Pkt Len Min': 'bidirectional_min_ps',
    'Bwd Pkt Len Mean': 'bidirectional_mean_ps',
    'Bwd Pkt Len Std': 'bidirectional_stddev_ps',
    'Flow Byts/s': 'Flow Byts/s',
    'Flow Pkts/s': 'Flow Pkts/s',
    'Flow IAT Mean': 'Flow IAT Mean',
    'Flow IAT Std': 'Flow IAT Std',
    'Flow IAT Max': 'Flow IAT Max',
    'Flow IAT Min': 'Flow IAT Min',
    'Fwd IAT Tot': 'Fwd IAT Tot',
    'Fwd IAT Mean': 'Fwd IAT Mean',
    'Fwd IAT Std': 'Fwd IAT Std',
    'Fwd IAT Max': 'Fwd IAT Max',
    'Fwd IAT Min': 'Fwd IAT Min',
    'Bwd IAT Tot': 'Bwd IAT Tot',
    'Bwd IAT Mean': 'Bwd IAT Mean',
    'Bwd IAT Std': 'Bwd IAT Std',
    'Bwd IAT Max': 'Bwd IAT Max',
    'Bwd IAT Min': 'Bwd IAT Min',
    'Fwd PSH Flags': 'Fwd PSH Flags',
    'Bwd PSH Flags': 'Bwd PSH Flags',
    'Fwd URG Flags': 'Fwd URG Flags',
    'Bwd URG Flags': 'Bwd URG Flags',
    'Fwd Header Len': 'Fwd Header Len',
    'Bwd Header Len': 'Bwd Header Len',
    'Fwd Pkts/s': 'Fwd Pkts/s',
    'Bwd Pkts/s': 'Bwd Pkts/s',
    'Pkt Len Min': 'Pkt Len Min',
    'Pkt Len Max': 'Pkt Len Max',
    'Pkt Len Mean': 'Pkt Len Mean',
    'Pkt Len Std': 'Pkt Len Std',
    'Pkt Len Var': 'Pkt Len Var',
    'FIN Flag Cnt': 'FIN Flag Cnt',
    'SYN Flag Cnt': 'SYN Flag Cnt',
    'RST Flag Cnt': 'RST Flag Cnt',
    'PSH Flag Cnt': 'PSH Flag Cnt',
    'ACK Flag Cnt': 'ACK Flag Cnt',
    'URG Flag Cnt': 'URG Flag Cnt',
    'CWE Flag Count': 'CWE Flag Count',
    'ECE Flag Cnt': 'ECE Flag Cnt',
    'Down/Up Ratio': 'Down/Up Ratio',
    'Pkt Size Avg': 'Pkt Size Avg',
    'Fwd Seg Size Avg': 'Fwd Seg Size Avg',
    'Bwd Seg Size Avg': 'Bwd Seg Size Avg',
    'Fwd Byts/b Avg': 'Fwd Byts/b Avg',
    'Fwd Pkts/b Avg': 'Fwd Pkts/b Avg',
    'Fwd Blk Rate Avg': 'Fwd Blk Rate Avg',
    'Bwd Byts/b Avg': 'Bwd Byts/b Avg',
    'Bwd Pkts/b Avg': 'Bwd Pkts/b Avg',
    'Bwd Blk Rate Avg': 'Bwd Blk Rate Avg',
    'Subflow Fwd Pkts': 'Subflow Fwd Pkts',
    'Subflow Fwd Byts': 'Subflow Fwd Byts',
    'Subflow Bwd Pkts': 'Subflow Bwd Pkts',
    'Subflow Bwd Byts': 'Subflow Bwd Byts',
    'Init Fwd Win Byts': 'Init Fwd Win Byts',
    'Init Bwd Win Byts': 'Init Bwd Win Byts',
    'Fwd Act Data Pkts': 'Fwd Act Data Pkts',
    'Fwd Seg Size Min': 'Fwd Seg Size Min',
    'Active Mean': 'Active Mean',
    'Active Std': 'Active Std',
    'Active Max': 'Active Max',
    'Active Min': 'Active Min',
    'Idle Mean': 'Idle Mean',
    'Idle Std': 'Idle Std',
    'Idle Max': 'Idle Max',
    'Idle Min': 'Idle Min',
}

# Appliquer le renommage
df_live.rename(columns=col_mapping, inplace=True)

# Supprimer les éventuelles colonnes dupliquées (si certaines ont été créées)
df_live = df_live.loc[:, ~df_live.columns.duplicated()]

# Garder uniquement les colonnes numériques
live_numeric = df_live.select_dtypes(include=[np.number]).columns.tolist()

print(f'   Numeric columns in test CSV  : {len(live_numeric)}')
print(f'   Columns used for training    : {len(feature_cols)}')

# ─── 6.3 Alignment with model features ─────────────────
print('\n─── Alignment with model features ───────────────')

common  = [c for c in feature_cols if c in live_numeric]
missing = [c for c in feature_cols if c not in live_numeric]
extra   = [c for c in live_numeric if c not in feature_cols]

print(f'   Common columns      : {len(common)}')
if missing:
    print(f'   Missing columns (→0): {len(missing)}')
    for m in missing[:5]:
        print(f'     - {m}')
    if len(missing) > 5:
        print(f'     ... and {len(missing)-5} others')
if extra:
    print(f'   Ignored columns (extra): {len(extra)}')

# Construire X_live avec exactement les mêmes colonnes que feature_cols
X_live_df = pd.DataFrame(index=df_live.index)
for col in feature_cols:
    if col in df_live.columns:
        X_live_df[col] = df_live[col]
    else:
        X_live_df[col] = 0

# ─── 6.4 Data cleaning ───────────────────────
print('\n─── Data cleaning ─────────────────────────────────')

inf_count = np.isinf(X_live_df.values).sum()
nan_count = X_live_df.isnull().sum().sum()
print(f'   Inf values found: {inf_count}')
print(f'   NaN values found: {nan_count}')

X_live_df = X_live_df.replace([np.inf, -np.inf], np.nan)
X_live_df = X_live_df.fillna(X_live_df.median())
print(f'   After cleaning: 0 inf, 0 NaN')

# ─── 6.5 Normalization with trained scaler ──────────────────
X_live = X_live_df.values.astype(float)
X_live_scaled = scaler.transform(X_live)
print(f'\n✅ Data normalized — shape: {X_live_scaled.shape}')

# ─── 6.6 Predictions with the 3 models ─────────────────────────
print('\n' + '=' * 60)
print('         PREDICTION RESULTS')
print('=' * 60)

all_preds = {}

for model_name, model, is_dl in [
    ('Random Forest', rf_model,  False),
    ('LightGBM',      lgb_model, False),
    ('Improved MLP',  mlp_v2,    True),
]:
    if is_dl:
        probs = model.predict(X_live_scaled, verbose=0)
        preds = np.argmax(probs, axis=1)
    else:
        preds = model.predict(X_live_scaled)
        probs = model.predict_proba(X_live_scaled)

    labels     = le.inverse_transform(preds)
    confs      = np.max(probs, axis=1) * 100
    is_attacks = ['spoof' in l.lower() or 'attack' in l.lower() or 'mitm' in l.lower()
                  for l in labels]
    n_attack   = sum(is_attacks)
    n_normal   = len(labels) - n_attack
    conf_avg   = confs.mean()
    conf_atk   = confs[is_attacks].mean() if n_attack > 0 else 0.0

    all_preds[model_name] = {'labels': labels, 'confs': confs, 'is_attacks': is_attacks}

    verdict = '🚨 ATTACK DETECTED' if n_attack > 0 else '✅ NORMAL TRAFFIC'
    print(f'\n  {model_name}')
    print(f'  {"─" * 45}')
    print(f'  Verdict          : {verdict}')
    print(f'  Analyzed flows   : {len(labels)}')
    print(f'  Attacks detected : {n_attack}  ({n_attack/len(labels)*100:.1f}%)')
    print(f'  Normal traffic   : {n_normal}  ({n_normal/len(labels)*100:.1f}%)')
    print(f'  Average confidence: {conf_avg:.1f}%')
    if n_attack > 0:
        print(f'  Attack confidence: {conf_atk:.1f}%')

# ─── 6.7 Majority vote + detailed table flow by flow ───────────────
print('\n' + '=' * 60)
print('   DETAILED TABLE — FLOW BY FLOW')
print('=' * 60)

votes = []
for i in range(len(df_live)):
    atk_votes = sum(all_preds[m]['is_attacks'][i] for m in all_preds)
    votes.append('🚨 ATTACK' if atk_votes >= 2 else '✅ NORMAL')

res_df = pd.DataFrame()
res_df['Flow#'] = range(1, len(df_live) + 1)
if 'Src IP' in df_live.columns:
    res_df['Src IP'] = df_live['Src IP'].values
if 'Dst IP' in df_live.columns:
    res_df['Dst IP'] = df_live['Dst IP'].values

for model_name, data in all_preds.items():
    short = model_name.split()[0][:3].upper()
    res_df[f'{short}_Result'] = ['🚨 Attack' if a else '✅ Normal' for a in data['is_attacks']]
    res_df[f'{short}_Confidence'] = [f'{c:.1f}%' for c in data['confs']]

res_df['Final Vote (2/3)'] = votes
display(res_df)

# ─── 6.8 Graphs ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Detection results — {os.path.basename(csv_live)}',
             fontsize=14, fontweight='bold')

model_colors = ['#2563eb', '#16a34a', '#7c3aed']
for idx, (model_name, data) in enumerate(all_preds.items()):
    ax = axes[idx]
    n_atk = sum(data['is_attacks'])
    n_nor = len(data['labels']) - n_atk

    if n_atk > 0:
        sizes      = [n_atk, n_nor]
        lbls_pie   = [f'Attack ({n_atk})', f'Normal ({n_nor})']
        clrs_pie   = ['#ef4444', '#10b981']
    else:
        sizes      = [n_nor]
        lbls_pie   = [f'Normal ({n_nor})']
        clrs_pie   = ['#10b981']

    wedges, texts, autotexts = ax.pie(
        sizes, labels=lbls_pie, colors=clrs_pie,
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
    )
    for at in autotexts:
        at.set_fontsize(10)
        at.set_fontweight('bold')
    ax.set_title(model_name, fontsize=12, fontweight='bold',
                 color=model_colors[idx])

plt.tight_layout()
plt.savefig('detection_live_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Confidence per flow — bar chart
fig2, ax2 = plt.subplots(figsize=(14, 4))
x_idx = np.arange(len(df_live))
w     = 0.28
for i, (model_name, data) in enumerate(all_preds.items()):
    ax2.bar(x_idx + i*w, data['confs'], w,
            label=model_name, color=model_colors[i], alpha=0.8)

ax2.set_xlabel('Flow index', fontsize=11)
ax2.set_ylabel('Confidence (%)', fontsize=11)
ax2.set_title('Confidence level per flow', fontsize=13, fontweight='bold')
ax2.axhline(y=80, color='gray', linestyle='--', alpha=0.5, label='Threshold 80%')
ax2.set_ylim(0, 110)
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('detection_live_confidence.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── 6.9 Final summary ───────────────────────────────────────────
consensus_attacks = sum(1 for v in votes if 'ATTACK' in v)
consensus_normal  = len(votes) - consensus_attacks

print('\n' + '=' * 60)
print('                   FINAL SUMMARY')
print('=' * 60)
print(f'  Analyzed file  : {os.path.basename(csv_live)}')
print(f'  Total flows    : {len(df_live)}')
print(f'  Final vote     : {consensus_attacks} attacks / {consensus_normal} normal')
print(f'  Attack rate    : {consensus_attacks/len(df_live)*100:.1f}%')
print()
if consensus_attacks > 0:
    print('  CONCLUSION: ARP Spoofing traffic detected!')
    print('  The 3 models (or 2/3) have identified suspicious flows.')
    print('  Check the suspicious IP addresses in the table above.')
else:
    print('  CONCLUSION: No attack detected.')
    print('  The captured traffic is classified as normal by the 3 models.')
print('=' * 60)
print('Graphs saved:')
print('   - detection_live_results.png')
print('   - detection_live_confidence.png')